In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Run:
  "C:\Program Files (x86)\MGLTools-1.5.7\python.exe" vina_pipeline.py
"""

from __future__ import print_function
import os, sys, shutil, subprocess


# ============================================================
#  ★ 여기만 수정 ★
# ============================================================

# ── 단백질 ──────────────────────────────────────────────────
# PDB ID로 자동 다운로드할 경우 -> PDB_ID에 입력, PDB_FILE은 ""
# 로컬 파일을 쓸 경우          -> PDB_FILE에 경로 입력, PDB_ID는 ""
PDB_ID   = ""
PDB_FILE = r"C:\autodockWork\46\tas2r46.pdb"

# ── Flexible Residues ────────────────────────────────────────
# Flexible residue 목록 (없으면 빈 리스트 [] 로 두세요)
# 포맷: "잔기이름:체인:번호"
FLEX_RESIDUES = [
    "TRP:R:88",
    "GLU:R:265",
]

# ── 리간드 ──────────────────────────────────────────────────
LIGAND_FILE = r"C:\autodockWork\strychnine.pdb"

# ── 도킹 모드 ────────────────────────────────────────────────
# "auto"    : ★ 권장 ★ explore → refine 자동 완전 수행
#             좌표 입력 불필요, 파일/residue만 설정하면 끝!
# "explore" : 1단계 탐색만 수행 (빠른 확인용)
# "refine"  : 2단계 정밀만 수행 (CENTER_X/Y/Z 필요)
DOCKING_MODE = "auto"

# ── pocket 크기에 추가할 여유 공간 (Å) ───────────────────────
GRID_BUFFER  = 5.0

# ── Grid Box 기준 좌표 (refine 단독 모드에서만 필요) ──────────
# auto / explore 모드에서는 사용하지 않습니다
CENTER_X =  0.0
CENTER_Y =  0.0
CENTER_Z =  0.0

# ── AutoDock Vina 실행 파일 경로 ─────────────────────────────
VINA_EXE = r"C:\Program Files (x86)\Vina\vina.exe"

# ── 출력 폴더 ────────────────────────────────────────────────
OUTPUT_DIR = r"C:\docking\results"

# ── 출력 파일명 접두사 ────────────────────────────────────────
# 비워두면 PDB 파일명이 자동으로 사용됩니다 (예: tas2r46)
OUTPUT_PREFIX = "tas2r46"

# ── 도킹 옵션 ─────────────────────────────────────────────────
EXHAUSTIVENESS_EXPLORE = 4     # 탐색용 (빠름)
EXHAUSTIVENESS_REFINE  = 12    # 정밀용 (정확)
EXPLORE_TOP_N          = 3     # explore에서 시도할 포켓 수 (상위 N개)
NUM_MODES              = 9
ENERGY_RANGE           = 3

# ============================================================
#  ▲ 수정 끝 ▲  아래는 돈터치!!
# ============================================================

MGLTOOLS     = r"C:\Program Files (x86)\MGLTools-1.5.7"
PYTHON_EXE   = os.path.join(MGLTOOLS, "python.exe")
SCRIPTS_DIR  = os.path.join(MGLTOOLS, "Lib", "site-packages", "AutoDockTools", "Utilities24")

PREP_RECEPTOR = os.path.join(SCRIPTS_DIR, "prepare_receptor4.py")
PREP_FLEX     = os.path.join(SCRIPTS_DIR, "prepare_flexreceptor4.py")
PREP_LIGAND   = os.path.join(SCRIPTS_DIR, "prepare_ligand4.py")


def log(msg):
    print("  >> " + msg)


def run_cmd(cmd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out, _ = proc.communicate()
    output = out.decode("utf-8", errors="replace").strip()
    if output:
        print(output)
    if proc.returncode != 0:
        print("[ERROR] Command failed: " + " ".join(cmd))
        sys.exit(1)


def win_to_wsl_path(win_path):
    """Windows 경로를 WSL 경로로 변환. 예: C:\\foo\\bar -> /mnt/c/foo/bar"""
    win_path = os.path.abspath(win_path)
    drive, rest = os.path.splitdrive(win_path)          # "C:", "\\foo\\bar"
    drive_letter = drive.rstrip(":").lower()             # "c"
    rest_posix = rest.replace("\\", "/")                 # "/foo/bar"
    return "/mnt/" + drive_letter + rest_posix           # "/mnt/c/foo/bar"


def run_fpocket(pdb_file):
    """WSL의 fpocket으로 실행 후 결과 디렉터리 반환."""
    log("Running fpocket via WSL: " + pdb_file)
    wsl_pdb = win_to_wsl_path(pdb_file)
    proc = subprocess.Popen(
        ["wsl", "--", "fpocket", "-f", wsl_pdb],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    out, _ = proc.communicate()
    if proc.returncode != 0:
        print("[ERROR] fpocket failed")
        print(out.decode("utf-8", errors="replace"))
        sys.exit(1)

    stem    = os.path.splitext(pdb_file)[0]
    out_dir = stem + "_out"
    if not os.path.isdir(out_dir):
        print("[ERROR] fpocket output directory not found: " + out_dir)
        sys.exit(1)
    log("fpocket output: " + out_dir)
    return out_dir


def _best_affinity(pdbqt_path):
    """결과 pdbqt에서 최고 affinity 반환. 없으면 0.0."""
    if not os.path.exists(pdbqt_path):
        return 0.0
    with open(pdbqt_path) as f:
        for line in f:
            if line.startswith("REMARK VINA RESULT:"):
                try:
                    return float(line.split()[3])
                except (IndexError, ValueError):
                    pass
    return 0.0


def calc_grid_box(pdb_file, flex_residues, fpocket_dir, best_pocket):
    """
    Flex residues(필수) + fpocket pocket(선택)의 union bounding box로
    grid box 중심/크기 계산. 두 조건을 만족하는 최소 크기.
    반환: (cx, cy, cz, sx, sy, sz)
    """
    # ── flex residue 원자 좌표 (PDB에서) ─────────────────────
    targets = set()
    for fr in flex_residues:
        parts = fr.split(":")
        if len(parts) == 3:
            targets.add((parts[0].strip(), parts[1].strip(), parts[2].strip()))

    flex_coords = []
    with open(pdb_file) as f:
        for line in f:
            if not (line.startswith("ATOM") or line.startswith("HETATM")):
                continue
            try:
                res_name = line[17:20].strip()
                chain    = line[21].strip()
                res_num  = line[22:26].strip()
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                if (res_name, chain, res_num) in targets:
                    flex_coords.append((x, y, z))
            except (ValueError, IndexError):
                pass

    if not flex_coords:
        log("[WARN] Flex residue coords not found in PDB, using pocket only")

    # ── fpocket alpha sphere 좌표 ─────────────────────────────
    pockets_dir = os.path.join(fpocket_dir, "pockets")
    pocket_coords = []
    vert_pqr = os.path.join(pockets_dir, "pocket%d_vert.pqr" % best_pocket)
    if os.path.exists(vert_pqr):
        with open(vert_pqr) as f:
            for line in f:
                if line.startswith("ATOM") or line.startswith("HETATM"):
                    try:
                        pocket_coords.append((
                            float(line[30:38]),
                            float(line[38:46]),
                            float(line[46:54]),
                        ))
                    except ValueError:
                        pass

    # ── union bounding box ────────────────────────────────────
    all_coords = flex_coords + pocket_coords
    if not all_coords:
        log("[WARN] No coords found, using default grid")
        return 0.0, 0.0, 0.0, 30.0, 30.0, 30.0

    xs = [c[0] for c in all_coords]
    ys = [c[1] for c in all_coords]
    zs = [c[2] for c in all_coords]

    cx = (max(xs) + min(xs)) / 2.0
    cy = (max(ys) + min(ys)) / 2.0
    cz = (max(zs) + min(zs)) / 2.0
    sx = (max(xs) - min(xs)) + GRID_BUFFER * 2
    sy = (max(ys) - min(ys)) + GRID_BUFFER * 2
    sz = (max(zs) - min(zs)) + GRID_BUFFER * 2

    log("Flex residue atoms : %d" % len(flex_coords))
    log("Pocket alpha spheres: %d" % len(pocket_coords))
    log("Grid box center    : (%.3f, %.3f, %.3f)" % (cx, cy, cz))
    log("Grid box size      : (%.1f, %.1f, %.1f) A" % (sx, sy, sz))
    return cx, cy, cz, sx, sy, sz


def parse_fpocket(fpocket_dir, pdb_stem, ref_cx=None, ref_cy=None, ref_cz=None,
                  pocket_num=None):
    """
    fpocket 4.x 결과에서 pocket 중심 좌표와 크기를 추출.
    - ref_cx/cy/cz 없으면 (explore): score 1위 pocket 자동 선택
    - ref_cx/cy/cz 있으면 (refine) : 기준 좌표에 가장 가까운 pocket 선택
    - 중심: pocket{N}_vert.pqr (alpha sphere) 좌표 평균
    - 크기: pocket{N}_atm.pdb bounding box + GRID_BUFFER
    반환: (cx, cy, cz, sx, sy, sz, pocket_num)
    """
    import math

    info_file   = os.path.join(fpocket_dir, pdb_stem + "_info.txt")
    pockets_dir = os.path.join(fpocket_dir, "pockets")

    if not os.path.exists(info_file):
        print("[ERROR] fpocket info file not found: " + info_file)
        sys.exit(1)

    # ── info.txt에서 pocket 번호/score 파싱 ──────────────────
    pockets = {}
    current = None
    with open(info_file) as f:
        for line in f:
            line = line.strip()
            if line.startswith("Pocket ") and line.endswith(":"):
                current = int(line.split()[1])
                pockets[current] = {}
            if current is None:
                continue
            if "Score :" in line:
                try:
                    pockets[current]["score"] = float(line.split(":")[-1].strip())
                except ValueError:
                    pass

    if not pockets:
        print("[ERROR] No pockets found by fpocket")
        sys.exit(1)

    log("fpocket found %d pocket(s)" % len(pockets))

    # ── PDB/PQR 좌표 읽기 헬퍼 ───────────────────────────────
    def read_coords(filepath):
        xs, ys, zs = [], [], []
        if not os.path.exists(filepath):
            return xs, ys, zs
        with open(filepath) as fh:
            for line in fh:
                if line.startswith("ATOM") or line.startswith("HETATM"):
                    try:
                        xs.append(float(line[30:38]))
                        ys.append(float(line[38:46]))
                        zs.append(float(line[46:54]))
                    except ValueError:
                        pass
        return xs, ys, zs

    # ── 각 pocket 중심: _vert.pqr (alpha sphere) 평균 ────────
    for num in pockets:
        vert_pqr = os.path.join(pockets_dir, "pocket%d_vert.pqr" % num)
        xs, ys, zs = read_coords(vert_pqr)
        if xs:
            pockets[num]["cx"] = sum(xs) / len(xs)
            pockets[num]["cy"] = sum(ys) / len(ys)
            pockets[num]["cz"] = sum(zs) / len(zs)

    # ── pocket 선택 ──────────────────────────────────────────
    if pocket_num is not None:
        # 특정 번호 지정 (auto explore 루프용)
        best = pocket_num
    elif ref_cx is None:
        # explore 모드: score 1위 pocket
        best = 1
        log("Explore mode: using highest-score pocket 1")
    else:
        # refine 모드: 기준 좌표에 가장 가까운 pocket
        log("Refine mode - reference: (%.3f, %.3f, %.3f)" % (ref_cx, ref_cy, ref_cz))
        best, best_dist = 1, float("inf")
        for num, p in pockets.items():
            if "cx" not in p:
                continue
            d = math.sqrt((p["cx"]-ref_cx)**2 + (p["cy"]-ref_cy)**2 + (p["cz"]-ref_cz)**2)
            if d < best_dist:
                best_dist, best = d, num
        log("Closest pocket to reference: pocket %d (dist=%.2f A)" % (best, best_dist))

    chosen = pockets[best]
    if "cx" not in chosen:
        print("[ERROR] Could not compute center for pocket %d" % best)
        sys.exit(1)
    cx, cy, cz = chosen["cx"], chosen["cy"], chosen["cz"]

    # ── 크기: _atm.pdb bounding box + buffer ─────────────────
    atm_pdb = os.path.join(pockets_dir, "pocket%d_atm.pdb" % best)
    xs, ys, zs = read_coords(atm_pdb)
    if xs:
        sx = (max(xs) - min(xs)) + GRID_BUFFER * 2
        sy = (max(ys) - min(ys)) + GRID_BUFFER * 2
        sz = (max(zs) - min(zs)) + GRID_BUFFER * 2
    else:
        sx = sy = sz = 20.0 + GRID_BUFFER * 2

    log("Pocket %d center : (%.3f, %.3f, %.3f)" % (best, cx, cy, cz))
    log("Grid box size    : (%.1f, %.1f, %.1f) A" % (sx, sy, sz))
    return cx, cy, cz, sx, sy, sz, best


def download_pdb(pdb_id, out_dir):
    out_path = os.path.join(out_dir, pdb_id.upper() + ".pdb")
    if os.path.exists(out_path):
        log("Already exists: " + out_path)
        return out_path
    url = "https://files.rcsb.org/download/" + pdb_id.upper() + ".pdb"
    log("Downloading: " + url)
    if sys.version_info[0] == 2:
        import urllib2
        data = urllib2.urlopen(url).read()
    else:
        import urllib.request
        data = urllib.request.urlopen(url).read()
    with open(out_path, "wb") as f:
        f.write(data)
    log("Saved: " + out_path)
    return out_path


def prepare_receptor(pdb_file, flex_residues, out_dir, prefix):
    temp_full = os.path.join(out_dir, prefix + "_full.pdbqt")
    rigid_out = os.path.join(out_dir, prefix + "_rigid.pdbqt")
    flex_out  = os.path.join(out_dir, prefix + "_flex.pdbqt")

    # Step 1: polar H + Gasteiger -> full PDBQT
    log("Adding polar hydrogens + computing Gasteiger charges ...")
    run_cmd([
        PYTHON_EXE, PREP_RECEPTOR,
        "-r", pdb_file,
        "-o", temp_full,
        "-A", "hydrogens",
        "-U", "nphs_lps_waters_nonstdres",
    ])
    log("Full receptor saved: " + temp_full)

    # Step 2: split into rigid + flex (별도 프로세스로 실행해 메모리 충돌 방지)
    if flex_residues:
        res_names = [r.split(":")[0] + r.split(":")[2] for r in flex_residues]
        log("Setting flexible residues: " + ", ".join(res_names))

        # prepare_flexreceptor4.py 에 버그 있음 (ResidueSet 결과를 res에 저장 안 함)
        # → 원본을 읽어서 한 줄만 패치한 임시 스크립트로 실행
        with open(PREP_FLEX, "r") as f:
            src = f.read()

        # 버그 수정: matched 가 ResidueSet일 때 res 에 저장하도록 추가
        old_code = (
            "                matched = rec.chains.residues.get(lambda x: x.name==n)\n"
            "                if matched.__class__ ==AtomSet:\n"
            "                    res = matched.parent.uniq() #@@\n"
            "                if len(res):"
        )
        new_code = (
            "                matched = rec.chains.residues.get(lambda x, _n=n: x.name==_n)\n"
            "                if matched.__class__ ==AtomSet:\n"
            "                    res = matched.parent.uniq() #@@\n"
            "                elif matched:\n"
            "                    res = matched\n"
            "                if len(res):"
        )
        if old_code not in src:
            print("[ERROR] prepare_flexreceptor4.py format changed, cannot patch")
            sys.exit(1)
        patched_src = src.replace(old_code, new_code)

        # 스레드 64MB 스택으로 실행해 스택 오버플로우 방지
        patched_src = (
            "import threading, sys\n"
            "def run():\n"
            "    from MolKit import Read\n"
            "    from MolKit.protein import ResidueSet, AtomSet\n"
            "    from AutoDockTools.MoleculePreparation import AD4FlexibleReceptorPreparation\n"
            "    rec = Read(r'" + temp_full.replace('\\','\\\\') + "')[0]\n"
            "    rec.buildBondsByDistance()\n"
            "    res_names = " + repr([r.split(':')[0]+r.split(':')[2] for r in flex_residues]) + "\n"
            "    all_res = ResidueSet()\n"
            "    for n in res_names:\n"
            "        matched = rec.chains.residues.get(lambda x, _n=n: x.name==_n)\n"
            "        if matched.__class__ == AtomSet:\n"
            "            all_res += matched.parent.uniq()\n"
            "        elif matched:\n"
            "            all_res += matched\n"
            "        else:\n"
            "            print('WARNING: not found: ' + n)\n"
            "    if not len(all_res):\n"
            "        print('ERROR: no residues found'); sys.exit(1)\n"
            "    print('Found ' + str(len(all_res)) + ' residue(s)')\n"
            "    AD4FlexibleReceptorPreparation(\n"
            "        rec, residues=all_res,\n"
            "        rigid_filename=r'" + rigid_out.replace('\\','\\\\') + "',\n"
            "        flexres_filename=r'" + flex_out.replace('\\','\\\\') + "',\n"
            "    )\n"
            "    print('done')\n"
            "threading.stack_size(64*1024*1024)\n"
            "t = threading.Thread(target=run)\n"
            "t.start()\n"
            "t.join()\n"
        )

        tmp_flex_script = os.path.join(out_dir, "_flex_prep.py")
        with open(tmp_flex_script, "w") as f:
            f.write(patched_src)

        res_names = [r.split(":")[0] + r.split(":")[2] for r in flex_residues]
        log("Running flex prep with large stack: " + "_".join(res_names))

        proc = subprocess.Popen(
            [PYTHON_EXE, tmp_flex_script],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )
        out, err = proc.communicate()
        stdout_txt = out.decode("utf-8", errors="replace").strip()
        stderr_txt = err.decode("utf-8", errors="replace").strip()
        if stdout_txt:
            print(stdout_txt)
        if stderr_txt:
            print(stderr_txt)
        os.remove(tmp_flex_script)
        if proc.returncode != 0:
            print("[ERROR] Flex preparation failed (exit code %d)" % proc.returncode)
            sys.exit(1)

        os.remove(temp_full)
        log("Rigid saved: " + rigid_out)
        log("Flex  saved: " + flex_out)
        return rigid_out, flex_out
    else:
        shutil.move(temp_full, rigid_out)
        log("Rigid saved: " + rigid_out)
        return rigid_out, None


def prepare_ligand(ligand_file, out_dir, prefix):
    out_path = os.path.join(out_dir, prefix + "_ligand.pdbqt")
    log("Adding H + computing Gasteiger charges ...")
    lig_dir  = os.path.dirname(os.path.abspath(ligand_file))
    lig_name = os.path.basename(ligand_file)
    # prepare_ligand4.py strips the directory, so run from ligand's folder
    proc = subprocess.Popen(
        [PYTHON_EXE, PREP_LIGAND, "-l", lig_name, "-o", out_path, "-A", "hydrogens"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        cwd=lig_dir,
    )
    out, _ = proc.communicate()
    output = out.decode("utf-8", errors="replace").strip()
    if output:
        print(output)
    if proc.returncode != 0:
        print("[ERROR] Ligand preparation failed")
        sys.exit(1)
    log("Ligand saved: " + out_path)
    return out_path


def write_config(out_dir, prefix, rigid, flex, ligand,
                  cx, cy, cz, sx, sy, sz, exhaustiveness):
    config_path = os.path.join(out_dir, prefix + "_config.txt")
    out_path    = os.path.join(out_dir, prefix + "_out.pdbqt")
    log_path    = os.path.join(out_dir, prefix + "_log.txt")

    lines = ["receptor = " + rigid]
    if flex:
        lines.append("flex     = " + flex)
    lines += [
        "ligand   = " + ligand,
        "",
        "center_x = %.3f" % cx,
        "center_y = %.3f" % cy,
        "center_z = %.3f" % cz,
        "",
        "size_x   = %.3f" % sx,
        "size_y   = %.3f" % sy,
        "size_z   = %.3f" % sz,
        "",
        "exhaustiveness = %d" % exhaustiveness,
        "num_modes      = %d" % NUM_MODES,
        "energy_range   = %d" % ENERGY_RANGE,
        "",
        "out = " + out_path,
        "log = " + log_path,
    ]
    with open(config_path, "w") as f:
        f.write("\n".join(lines) + "\n")

    log("Config saved: " + config_path)
    return config_path, out_path, log_path


def run_vina(config_path):
    log("Running AutoDock Vina ...")
    proc = subprocess.Popen(
        [VINA_EXE, "--config", config_path],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    out, _ = proc.communicate()
    print(out.decode("utf-8", errors="replace"))
    if proc.returncode != 0:
        print("[ERROR] Vina failed")
        sys.exit(1)


def print_results(out_pdbqt):
    if not os.path.exists(out_pdbqt):
        print("  [!] Result file not found: " + out_pdbqt)
        return
    results = []
    with open(out_pdbqt) as f:
        for line in f:
            if line.startswith("REMARK VINA RESULT:"):
                p = line.split()
                try:
                    results.append((float(p[3]), float(p[4]), float(p[5])))
                except (IndexError, ValueError):
                    pass
    print()
    print("  Mode | Affinity (kcal/mol) | RMSD lb | RMSD ub")
    print("  " + "-" * 46)
    for i, (aff, lb, ub) in enumerate(results, 1):
        print("  %4d |       %8.1f      | %7.3f | %7.3f" % (i, aff, lb, ub))
    if results:
        print("\n  Best pose binding energy: %.1f kcal/mol" % results[0][0])


def main():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    if OUTPUT_PREFIX:
        prefix = OUTPUT_PREFIX
    elif PDB_ID:
        prefix = PDB_ID.upper()
    else:
        prefix = os.path.splitext(os.path.basename(PDB_FILE))[0]

    print()
    print("=" * 50)
    print("  AutoDock Vina Docking  |  %s  [%s]" % (prefix, DOCKING_MODE.upper()))
    print("=" * 50)

    # ── [1] 단백질 PDB ───────────────────────────────────────
    print("\n[1/6] Protein PDB")
    pdb_file = download_pdb(PDB_ID, OUTPUT_DIR) if PDB_ID else PDB_FILE
    pdb_stem = os.path.splitext(os.path.basename(pdb_file))[0]

    # ── [2] 수용체 준비 ──────────────────────────────────────
    print("\n[2/6] Receptor preparation")
    rigid, flex = prepare_receptor(pdb_file, FLEX_RESIDUES, OUTPUT_DIR, prefix)

    # ── [3] 리간드 준비 ──────────────────────────────────────
    print("\n[3/6] Ligand preparation")
    ligand = prepare_ligand(LIGAND_FILE, OUTPUT_DIR, prefix)

    # ── [4] fpocket 포켓 탐지 ────────────────────────────────
    print("\n[4/6] Pocket detection (fpocket)")
    fpocket_dir = run_fpocket(pdb_file)

    # fpocket 결과 폴더를 OUTPUT_DIR로 복사
    dest_fpocket = os.path.join(OUTPUT_DIR, prefix + "_fpocket")
    if os.path.exists(dest_fpocket):
        shutil.rmtree(dest_fpocket)
    shutil.copytree(fpocket_dir, dest_fpocket)
    log("fpocket results : " + dest_fpocket)

    # ── [5] 도킹 수행 ────────────────────────────────────────
    if DOCKING_MODE == "auto":
        # ── 5-A: top N 포켓에 빠른 explore 도킹 ────────────
        print("\n[5/6] Docking  [STEP 1/2 : EXPLORE - top %d pockets]" % EXPLORE_TOP_N)

        best_affinity = float("inf")
        best_cx = best_cy = best_cz = 0.0
        best_sx = best_sy = best_sz = 24.0
        best_pocket = 1
        exp_pdbqt = None

        for pocket_num in range(1, EXPLORE_TOP_N + 1):
            log("--- Trying pocket %d / %d ---" % (pocket_num, EXPLORE_TOP_N))
            # flex residues(필수) + pocket(선택)의 최소 union bounding box
            cx, cy, cz, sx, sy, sz = calc_grid_box(
                pdb_file, FLEX_RESIDUES, fpocket_dir, pocket_num)

            exp_prefix = prefix + "_explore_p%d" % pocket_num
            config, pdbqt, _ = write_config(
                OUTPUT_DIR, exp_prefix, rigid, flex, ligand,
                cx, cy, cz, sx, sy, sz, EXHAUSTIVENESS_EXPLORE)
            run_vina(config)

            # 가장 좋은 affinity 확인
            affinity = _best_affinity(pdbqt)
            log("Pocket %d best affinity: %.2f kcal/mol" % (pocket_num, affinity))
            if affinity < best_affinity:
                best_affinity = affinity
                best_cx, best_cy, best_cz = cx, cy, cz
                best_sx, best_sy, best_sz = sx, sy, sz
                best_pocket = pocket_num
                exp_pdbqt = pdbqt

        log("==> Best pocket: %d (affinity %.2f kcal/mol)" % (best_pocket, best_affinity))
        cx, cy, cz = best_cx, best_cy, best_cz
        sx, sy, sz = best_sx, best_sy, best_sz
        print("  -- Best explore result (pocket %d) --" % best_pocket)
        print_results(exp_pdbqt)

        # ── 5-B: 최적 포켓으로 정밀 refine ──────────────────
        print("\n[5/6] Docking  [STEP 2/2 : REFINE - pocket %d]" % best_pocket)
        log("Exhaustiveness: %d → %d" % (EXHAUSTIVENESS_EXPLORE, EXHAUSTIVENESS_REFINE))

        ref_prefix = prefix + "_refine"
        config, out_pdbqt, log_path = write_config(
            OUTPUT_DIR, ref_prefix, rigid, flex, ligand,
            cx, cy, cz, sx, sy, sz, EXHAUSTIVENESS_REFINE)
        run_vina(config)

    elif DOCKING_MODE == "explore":
        print("\n[5/6] Docking  [EXPLORE]")
        best_pocket = 1
        cx, cy, cz, sx, sy, sz = calc_grid_box(
            pdb_file, FLEX_RESIDUES, fpocket_dir, best_pocket)
        config, out_pdbqt, log_path = write_config(
            OUTPUT_DIR, prefix, rigid, flex, ligand,
            cx, cy, cz, sx, sy, sz, EXHAUSTIVENESS_EXPLORE)
        run_vina(config)

    else:  # refine (CENTER_X/Y/Z 기준 가장 가까운 포켓)
        print("\n[5/6] Docking  [REFINE]")
        _, _, _, _, _, _, best_pocket = parse_fpocket(
            fpocket_dir, pdb_stem, CENTER_X, CENTER_Y, CENTER_Z)
        cx, cy, cz, sx, sy, sz = calc_grid_box(
            pdb_file, FLEX_RESIDUES, fpocket_dir, best_pocket)
        config, out_pdbqt, log_path = write_config(
            OUTPUT_DIR, prefix, rigid, flex, ligand,
            cx, cy, cz, sx, sy, sz, EXHAUSTIVENESS_REFINE)
        run_vina(config)

    # ── 포켓 요약 파일 저장 ──────────────────────────────────
    summary_path = os.path.join(OUTPUT_DIR, prefix + "_pocket_summary.txt")
    with open(summary_path, "w") as f:
        f.write("Mode            : %s\n" % DOCKING_MODE)
        f.write("Selected pocket : %d\n" % best_pocket)
        f.write("Center X        : %.3f\n" % cx)
        f.write("Center Y        : %.3f\n" % cy)
        f.write("Center Z        : %.3f\n" % cz)
        f.write("Size X          : %.3f\n" % sx)
        f.write("Size Y          : %.3f\n" % sy)
        f.write("Size Z          : %.3f\n" % sz)
    log("Pocket summary  : " + summary_path)

    # ── [6] 최종 결과 출력 ───────────────────────────────────
    print("\n[6/6] Final Results")
    print_results(out_pdbqt)

    print()
    print("=" * 50)
    print("  Done!")
    if DOCKING_MODE == "auto":
        print("  Explore : " + exp_pdbqt)
        print("  Refine  : " + out_pdbqt)
    else:
        print("  Result  : " + out_pdbqt)
    print("  Log     : " + log_path)
    print("=" * 50)


if __name__ == "__main__":
    main()
